In [5]:
from pathlib import Path

import pandas as pd

!noglob scp -r hkqai:~/workdir/cc2cc_test5/validate/*.csv ~/workspace/2025.1/validate
name_mol_list = [
    "molecule_W4_11",
    "molecule_G21EA",
    "molecule_G21IP",
    "molecule_DIPCS10",
    "molecule_PA26",
    "molecule_SIE4x4",
    "molecule_ALKBDE10",
    "molecule_YBDE18",
    "molecule_AL2X6",
    "molecule_HEAVYSB11",
    "molecule_NBPRC",
    "molecule_ALK8",
    "molecule_RC21",
    "molecule_G2RC",
    "molecule_BH76",
    "molecule_FH51",
    "molecule_TAUT15",
    "molecule_DC13",
    "molecule_MB16_43",
    "molecule_DARC",
    "molecule_RSE43",
    "molecule_BSR36",
    "molecule_CDIE20",
    "molecule_ISO34",
    "molecule_ISOL24",
    "molecule_C60ISO",
    "molecule_PArel",
    "molecule_BHPERI",
    "molecule_BHDIV10",
    "molecule_INV24",
    "molecule_BHROT27",
    "molecule_PX13",
    "molecule_WCPT18",
    "molecule_RG18",
    "molecule_ADIM6",
    "molecule_S22",
    "molecule_S66",
    "molecule_HEAVY28",
    "molecule_WATER27",
    "molecule_CARBHB12",
    "molecule_PNICO23",
    "molecule_HAL59",
    "molecule_AHB21",
    "molecule_CHB6",
    "molecule_IL16",
    "molecule_IDISP",
    "molecule_ICONF",
    "molecule_ACONF",
    "molecule_Amino20x4",
    "molecule_PCONF21",
    "molecule_MCONF",
    "molecule_SCONF",
    "molecule_UPU23",
    "molecule_BUT14DIOL",
]

if_start_file = {}

for i, name_mol in enumerate(name_mol_list):
    data_path_list = sorted(
        list(Path("../validate").glob(f"*{name_mol}.csv")),
        key=lambda p: p.stat().st_ctime,
    )
    
    for data_path in data_path_list:
        summary_data_path = data_path.stem.split("_" + name_mol)[0] + ".csv"

        with open(data_path, "r") as f:
            data = pd.read_csv(f)

        if if_start_file.get(summary_data_path, True):
            with open(Path("../validate") / summary_data_path, "w") as f2:
                data.to_csv(f2, index=False)
            if_start_file[summary_data_path] = False
        else:
            with open(Path("../validate") / summary_data_path, "a") as f2:
                data.to_csv(f2, index=False, header=False)

for i, name_mol in enumerate(name_mol_list):
    for data_path in list(Path("../validate").glob(f"*{name_mol}.csv")):
        data_path.unlink()

for data_path in list(Path("../validate").glob(f"*test*.csv")):
    data_path.unlink()

for summary_data_path in list(Path("../validate").glob("*-cc-pVDZ.csv")):
    with open(summary_data_path, "r") as f:
        data = pd.read_csv(f)

    with open(Path("../validate") / summary_data_path.name.replace("-cc-pVDZ", ""), "w") as f2:
        data.to_csv(f2, index=False)

ccdft_cc-pVDZ_atom-1-test_gmtkn-cc-pVDZ_molec 100% 8736     6.3MB/s   00:00    
ccdft_cc-pVDZ_atom-1-test_gmtkn-cc-pVDZ_molec 100% 7778   181.1KB/s   00:00    
ccdft_cc-pVDZ_atom-1-test_gmtkn-cc-pVDZ_molec 100% 5499   127.5KB/s   00:00    


In [9]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*{data_set}.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"../cc2cc/utils/{data_set}.json") as f:
    json_data = json.load(f)


full_subset_dict = {
    # "test": ["WATER27"],
    "sub1": [
        "W4_11",
        "G21EA",
        "G21IP",
        "DIPCS10",
        "PA26",
        "SIE4x4",
        "ALKBDE10",
        "YBDE18",
        "AL2X6",
        "HEAVYSB11",
        "NBPRC",
        "ALK8",
        "RC21",
        "G2RC",
        "BH76RC",
        "FH51",
        "TAUT15",
        "DC13",
    ],
    "sub2": [
        "MB16_43",
        "DARC",
        "RSE43",
        "BSR36",
        "CDIE20",
        "ISO34",
        "ISOL24",
        "C60ISO",
        "PArel",
    ],
    "sub3": [
        "BH76",
        "BHPERI",
        "BHDIV10",
        "INV24",
        "BHROT27",
        "PX13",
        "WCPT18",
    ],
    "sub4": [
        "RG18",
        "ADIM6",
        "S22",
        "S66",
        "HEAVY28",
        "WATER27",
        "CARBHB12",
        "PNICO23",
        "HAL59",
        "AHB21",
        "CHB6",
        "IL16",
    ],
    "sub5": [
        "IDISP",
        "ICONF",
        "ACONF",
        "Amino20x4",
        "PCONF21",
        "MCONF",
        "SCONF",
        "UPU23",
        "BUT14DIOL",
    ],
}
subset_list = []
for i_subset in full_subset_dict.keys():
    subset_list.extend([f"{i_subset}_{ i_set}" for i_set in full_subset_dict[i_subset]])

# accumulate summary dictionaries for each file
summary_list = []
subset_summary_list = []

for name_set, subset_list_ in full_subset_dict.items():
    summary_data_list = {}
    for i_subset in subset_list_:
        summary = {}

        for data_path in data_path_list:
            data = pd.read_csv(data_path)
            data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

            data_name = []
            data_reaction_energy_dft = []
            data_reaction_energy_ai = []

            reaction_dict = json_data[f"reaction-{i_subset}"]
            reaction_dict_copy = reaction_dict.copy()
            for i_reaction_name, i_reaction in reaction_dict_copy.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_ai = 0
                for i in range(len(systems_list)):
                    finished = True
                    mole_name = (
                        f"{i_subset}-{systems_list[i]}"
                        if i_subset != "BH76RC"
                        else systems_list[i]
                    )

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished = False
                        reaction_dict.pop(i_reaction_name)
                        break

                    col = data["name"] == mole_name
                    if col.any():
                        atomic_energy_dft += data[col]["error_dft_ene"].values[0] * int(
                            stoichiometry_list[i]
                        )
                        atomic_energy_ai += data[col]["error_scf_ene"].values[0] * int(
                            stoichiometry_list[i]
                        )
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if finished:
                    data_reaction_energy_dft.append(atomic_energy_dft)
                    data_reaction_energy_ai.append(atomic_energy_ai)
                    data_name.append(i_reaction_name)

            data_name = np.array(data_name)
            data_reaction_energy_dft = np.array(data_reaction_energy_dft)
            data_reaction_energy_ai = np.array(data_reaction_energy_ai)

            if verbose > 0:
                dft_error_argsort = np.argsort(np.abs(data_reaction_energy_dft))[::-1][:5]
                ai_error_argsort = np.argsort(np.abs(data_reaction_energy_ai))[::-1][:5]

                print(f"####data_path:{data_path}####")
                
                # print(f"====DFT error of {i_subset}====")
                # print(
                #     [
                #         json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                #         for i in dft_error_argsort
                #     ]
                # )
                # print(data_reaction_energy_dft[dft_error_argsort])
                # if verbose == 2:
                #     print(f"====Detail of DFT error of {i_subset}====")
                #     for i in dft_error_argsort:
                #         print(f"{data_name[i]}: {data_reaction_energy_dft[i]}")
                #         systems_list = json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                #         stoichiometry_list = json_data[f"reaction-{i_subset}"][data_name[i]]["stoichiometry"]
                #         for j in range(len(systems_list)):
                #             mole_name = (
                #                 f"{i_subset}-{systems_list[j]}"
                #                 if i_subset != "BH76RC"
                #                 else systems_list[j]
                #             )

                #             if mole_name in json_data:
                #                 if isinstance(json_data[mole_name], str):
                #                     mole_name = json_data[mole_name]

                #             col = data["name"] == mole_name
                #             print(
                #                 data[col]["error_dft_ene"].values[0],
                #                 int(stoichiometry_list[j]),
                #                 systems_list[j],
                #             )
                #         print()
                            
                print(f"====AI error of {i_subset}====")
                print(
                    [
                        json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                        for i in ai_error_argsort
                    ]
                )
                print(data_reaction_energy_ai[ai_error_argsort])
                if verbose == 2:
                    print(f"====Detail of AI error of {i_subset}====")
                    for i in ai_error_argsort:
                        print(f"{data_name[i]}: {data_reaction_energy_ai[i]}")
                        systems_list = json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                        stoichiometry_list = json_data[f"reaction-{i_subset}"][data_name[i]]["stoichiometry"]
                        for j in range(len(systems_list)):
                            mole_name = (
                                f"{i_subset}-{systems_list[j]}"
                                if i_subset != "BH76RC"
                                else systems_list[j]
                            )

                            if mole_name in json_data:
                                if isinstance(json_data[mole_name], str):
                                    mole_name = json_data[mole_name]

                            col = data["name"] == mole_name
                            print(
                                data[col]["error_scf_ene"].values[0],
                                int(stoichiometry_list[j]),
                                systems_list[j],
                            )
                        print()
                            
            summary.update(
                {
                    f"{data_path.stem} AI AE": (
                        np.mean(np.abs(data_reaction_energy_ai))
                        if len(data_reaction_energy_ai)
                        else 0
                    ),
                    f"{data_path.stem} DFT AE": (
                        np.mean(np.abs(data_reaction_energy_dft))
                        if len(data_reaction_energy_dft)
                        else 0
                    ),
                    f"{data_path.stem} Processed": f"{len(data_reaction_energy_dft)} / {len(reaction_dict)}",
                }
            )

            if f"{data_path.stem} AI AE" not in summary_data_list:
                summary_data_list[f"{data_path.stem} AI AE"] = data_reaction_energy_ai
            else:
                summary_data_list[f"{data_path.stem} AI AE"] = np.append(
                    summary_data_list[f"{data_path.stem} AI AE"],
                    data_reaction_energy_ai,
                )

            if f"{data_path.stem} DFT AE" not in summary_data_list:
                summary_data_list[f"{data_path.stem} DFT AE"] = data_reaction_energy_dft
            else:
                summary_data_list[f"{data_path.stem} DFT AE"] = np.append(
                    summary_data_list[f"{data_path.stem} DFT AE"],
                    data_reaction_energy_dft,
                )

            if f"{data_path.stem} reaction_dict" not in summary_data_list:
                summary_data_list[f"{data_path.stem} reaction_dict"] = len(
                    reaction_dict
                )
            else:
                summary_data_list[f"{data_path.stem} reaction_dict"] += len(
                    reaction_dict
                )

        summary_list.append(summary)

    subset_summary = {}
    for data_path in data_path_list:
        subset_summary.update(
            {
                f"{data_path.stem} AI AE": (
                    np.mean(np.abs(summary_data_list[f"{data_path.stem} AI AE"]))
                    if len(summary_data_list[f"{data_path.stem} AI AE"])
                    else 0
                ),
                f"{data_path.stem} DFT AE": (
                    np.mean(np.abs(summary_data_list[f"{data_path.stem} DFT AE"]))
                    if len(summary_data_list[f"{data_path.stem} DFT AE"])
                    else 0
                ),
                f"{data_path.stem} Processed": f"{len(summary_data_list[f"{data_path.stem} AI AE"])} / {summary_data_list[f"{data_path.stem} reaction_dict"]}",
            }
        )
    subset_summary_list.append(subset_summary)

# display one summary table for all files
header = pd.MultiIndex.from_product(
    [
        [data_path.stem for data_path in data_path_list],
        ["AI AE", "DFT AE", "Processed"],
    ],
    names=["data_path", "Type"],
)
df_summary = pd.DataFrame(
    subset_summary_list,
    index=full_subset_dict.keys(),
)
df_summary.columns = header
display(df_summary)

# save summary to csv with date
df_summary.to_csv(f"../validate/summary_set_{date}.csv")

header = pd.MultiIndex.from_product(
    [
        [data_path.stem for data_path in data_path_list],
        ["AI AE", "DFT AE", "Processed"],
    ],
    names=["data_path", "Type"],
)
df_summary = pd.DataFrame(
    summary_list,
    index=subset_list,
)
df_summary.columns = header

df_summary.sort_values(
    by=("ccdft_cc-pVDZ_atom-1-3881456_4750_gmtkn-cc-pVDZ", "AI AE"),
    axis=0,
    inplace=True,
    ascending=False,
)

display(df_summary)

# save summary to csv with date
df_summary.to_csv(f"../validate/summary_subset_{date}.csv")

cc-pVDZ
####data_path:../validate/ccdft_cc-pVDZ_atom-1-1150169_gmtkn-cc-pVDZ.csv####
====AI error of W4_11====
[['propane', 'c', 'h'], ['propene', 'c', 'h'], ['s4-c2v', 's'], ['acetic', 'c', 'o', 'h'], ['glyoxal', 'c', 'o', 'h']]
[ 19.41995005  17.52912642 -16.80355977  16.2599607   15.20639248]
====Detail of AI error of W4_11====
15: 19.41995004655437
-0.4830054018798897 -1 propane
0.5723009303366778 3 c
2.152505231708056 8 h

23: 17.5291264151846
-2.897192233926232 -1 propene
0.5723009303366778 3 c
2.152505231708056 6 h

128: -16.80355976843325
12.369090028304717 -1 s4-c2v
-1.108617435032133 4 s

52: 16.259960702272796
-2.896038679013616 -1 acetic
0.5723009303366778 2 c
1.8046496178768003 2 o
2.152505231708056 4 h

66: 15.206392481996781
-6.147480922153714 -1 glyoxal
0.5723009303366778 2 c
1.8046496178768003 2 o
2.152505231708056 2 h

####data_path:../validate/ccdft_cc-pVDZ_atom-1-3881456_4750_gmtkn-cc-pVDZ.csv####
====AI error of W4_11====
[['p4', 'p'], ['s4-c2v', 's'], ['sif4', 'si

data_path ccdft_cc-pVDZ_atom-1-1150169_gmtkn-cc-pVDZ                        \
Type                                           AI AE     DFT AE  Processed   
sub1                                        4.692939  18.629823  468 / 468   
sub2                                        6.272359   6.562349  222 / 243   
sub3                                        4.162538   6.309849  192 / 194   
sub4                                        2.254974   2.887648  242 / 244   
sub5                                        1.913818   1.245124  259 / 291   

data_path ccdft_cc-pVDZ_atom-1-3881456_4750_gmtkn-cc-pVDZ             \
Type                                                AI AE     DFT AE   
sub1                                             3.601181  18.629823   
sub2                                             5.700007   6.612165   
sub3                                             3.648439   6.764537   
sub4                                             2.277669   1.651300   
sub5                                             1.775151   1.321288   

data_path            ccdft_cc-pVDZ_atom-1-1918154_gmtkn-cc-pVDZ             \
Type       Processed                                      AI AE     DFT AE   
sub1       468 / 468                                   2.700498  24.176867   
sub2       210 / 243                                   0.000000   0.000000   
sub3       170 / 194                                   0.000000   0.000000   
sub4       217 / 244                                   0.000000   0.000000   
sub5       268 / 291                                   0.000000   0.000000   

data_path            
Type      Processed  
sub1       91 / 468  
sub2        0 / 243  
sub3        0 / 194  
sub4        0 / 244  
sub5        0 / 291

data_path      ccdft_cc-pVDZ_atom-1-1150169_gmtkn-cc-pVDZ             \
Type                                                AI AE     DFT AE   
sub2_MB16_43                                    11.357448  15.461608   
sub5_IDISP                                      15.704176  13.632636   
sub1_ALKBDE10                                    5.435985  18.125250   
sub1_DC13                                        9.695545  13.090863   
sub2_DARC                                        4.818050  10.754152   
sub1_YBDE18                                      7.255829   8.145395   
sub3_PX13                                       13.059142  11.042025   
sub1_AL2X6                                       5.084741   5.659146   
sub1_ALK8                                        6.925821   4.400063   
sub1_RC21                                        4.696462   4.820927   
sub1_SIE4x4                                     17.182399  21.908521   
sub1_NBPRC                                       2.745689   2.232550   
sub1_HEAVYSB11                                   3.928191   5.391477   
sub4_ADIM6                                       3.649860   3.059030   
sub3_BH76                                        4.203748   9.107948   
sub3_WCPT18                                      4.243739   7.967153   
sub1_DIPCS10                                     4.414108  12.310855   
sub3_BHDIV10                                     2.461823   6.277708   
sub1_G2RC                                        3.044318   5.917204   
sub1_PA26                                        3.011796   2.200767   
sub4_CHB6                                        3.585377   1.805320   
sub5_PCONF21                                     3.418177   3.683764   
sub4_S22                                         4.086915   2.291253   
sub1_FH51                                        3.055128   3.703658   
sub4_AHB21                                       1.766602   2.453821   
sub1_W4_11                                       5.805564  29.506869   
sub2_ISO34                                       2.057023   2.001744   
sub2_BSR36                                      15.616911   8.401182   
sub2_RSE43                                       1.951595   3.157601   
sub4_S66                                         2.331477   1.894710   
sub4_HAL59                                       1.848147   1.649787   
sub3_BHPERI                                      2.119660   3.268993   
sub1_TAUT15                                      1.871571   2.164531   
sub4_IL16                                        1.735752   1.021067   
sub2_CDIE20                                      1.160373   1.599507   
sub1_G21EA                                       2.127111   9.754524   
sub2_PArel                                       1.962423   1.743933   
sub1_G21IP                                       2.390142   8.953336   
sub5_ACONF                                       0.548364   0.546621   
sub5_ICONF                                       0.709620   0.413804   
sub5_Amino20x4                                   1.405904   0.656235   
sub3_BHROT27                                     1.078680   0.853379   
sub5_MCONF                                       3.392493   1.956021   
sub1_BH76RC                                      1.212571  80.220383   
sub5_SCONF                                       1.591377   0.512352   
sub4_CARBHB12                                    1.364792   1.632408   
sub5_BUT14DIOL                                   0.783503   0.645099   
sub4_PNICO23                                     0.717968   0.776429   
sub4_RG18                                        0.503217   0.230018   
sub2_C60ISO                                      0.000000   0.000000   
sub3_INV24                                       5.668748   2.796356   
sub4_HEAVY28                                     0.000000   0.000000   
sub2_ISOL24                                      4.843230   5.690557   
sub4_WATER27                            

In [ ]:
Error_molecule = [
    "G21IP-IP_80_cc-pVDZ_0-1_1_0.0000_default",
    "G21EA-EA_23n_cc-pVDZ_0-1_1_0.0000_default",
    "BH76-RKT15_cc-pVDZ_0-1_1_0.0000_default",
]
print(len(Error_molecule))

# mol_name =
# Error molecule: ['MB16_43-26_cc-pVDZ_0-1_1_0.5000_default', 'MB16_43-36_cc-pVDZ_0-1_1_-0.5000_default', 'BSR36-h11_cc-pVDZ_0-1_1_0.5000_default', 'BSR36-h7_cc-pVDZ_0-1_1_-0.5000_default']

str_print = ''
for molecule in Error_molecule:
    str_print += molecule.split("_cc-pVDZ_")[0] + " "
print(str_print)

3
G21IP-IP_80 G21EA-EA_23n BH76-RKT15 
